In [ ]:
import lmdb
import pickle

env = lmdb.open(
    "targetdiff/crossdocked_v1.1_rmsd1.0_pocket10_processed_final.lmdb",
    subdir=False,
    readonly=True,
    lock=False
)

with env.begin() as txn:
    cursor = txn.cursor()
    
    for key, value in cursor:
        sample = pickle.loads(value)
        
        print("Keys:", sample.keys())
        break

In [ ]:
import lmdb
import pickle
from tqdm import tqdm

def read_lmdb_dataset(lmdb_path, max_samples=None):
    env = lmdb.open(
        lmdb_path,
        subdir=False,
        readonly=True,
        lock=False,
        readahead=False,
        meminit=False
    )

    data = []

    with env.begin() as txn:
        cursor = txn.cursor()

        for i, (key, value) in enumerate(tqdm(cursor)):
            if max_samples and i >= max_samples:
                break

            sample = pickle.loads(value)

            smiles = sample.get("ligand_smiles", None)
            protein_file = sample.get("protein_filename", None)

            # حذف موارد خراب
            if smiles is None or len(smiles) == 0:
                continue

            data.append({
                "smiles": smiles,
                "protein_file": protein_file
            })

    env.close()
    return data


# تست
dataset = read_lmdb_dataset(
    "targetdiff/crossdocked_v1.1_rmsd1.0_pocket10_processed_final.lmdb",
    max_samples=None
)

print("Number of samples:", len(dataset))
print(dataset[0])

In [ ]:
pip install biopython

In [ ]:
#Extract sequence from pdb
from Bio.PDB import PDBParser, PPBuilder
import os

parser = PDBParser(QUIET=True)
ppb = PPBuilder()

def pdb_to_sequence(pdb_path):
    try:
        structure = parser.get_structure("protein", pdb_path)

        sequences = []
        for pp in ppb.build_peptides(structure):
            sequences.append(str(pp.get_sequence()))

        if len(sequences) == 0:
            return None

        return "".join(sequences)

    except:
        return None

In [ ]:
import torch
split = torch.load("targetdiff/crossdocked_pocket10_pose_split.pt")
train_idx = set(split["train"])
val_idx = set(split["val"])
test_idx = set(split["test"])

train_raw, val_raw, test_raw = [], [], []

for sample in dataset:
    idx = sample["index"]

    if idx in train_idx:
        train_raw.append(sample)
    elif idx in val_idx:
        val_raw.append(sample)
    elif idx in test_idx:
        test_raw.append(sample)

In [ ]:
print(len(train_raw), len(val_raw), len(test_raw))

In [ ]:
base_dir = "targetdiff/crossdocked_pocket10_with_protein"
train_data = build_dataset_from_pdb(train_raw, base_dir)
val_data = build_dataset_from_pdb(val_raw, base_dir)
test_data = build_dataset_from_pdb(test_raw, base_dir)

In [ ]:
torch.save(train_data, "train.pt")
torch.save(val_data, "val.pt")
torch.save(test_data, "test.pt")

In [ ]:
fail_count = 0

for sample in test_raw[:1000]:
    pdb_path = os.path.join("targetdiff/crossdocked_pocket10_with_protein", sample["protein_file"])
    
    seq = pdb_to_sequence(pdb_path)
    
    if seq is None:
        fail_count += 1

print("fail rate:", fail_count)

# ProtT5 Embeddings

In [ ]:
import torch
from transformers import T5Tokenizer, T5EncoderModel
from tqdm import tqdm
import re

In [ ]:
from transformers import T5EncoderModel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name="Rostlab/prot_t5_xl_uniref50"
tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)
model = T5EncoderModel.from_pretrained(model_name)

model = model.to(device)
model = model.eval()

In [ ]:
########################################
# preprocess
########################################
def preprocess_sequence(seq, max_len=1024):
    seq = seq.replace(" ", "")
    seq = re.sub(r"[UZOB]", "X", seq)
    seq = seq[:max_len]
    return " ".join(list(seq))

########################################
# embedding گرفتن
########################################
def embed_dataset(dataset, batch_size=8):
    embeddings = []
    smiles_list = []
    index_list = []

    for i in tqdm(range(0, len(dataset), batch_size)):
        batch = dataset[i:i+batch_size]

        seqs = [preprocess_sequence(s["protein_seq"]) for s in batch]
        smiles = [s["smiles"] for s in batch]
        idxs = [s["index"] for s in batch]

        inputs = tokenizer(seqs, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        emb = outputs.last_hidden_state

        # mean pooling
        mask = attention_mask.unsqueeze(-1).expand(emb.size()).float()
        summed = torch.sum(emb * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        pooled = summed / counts

        embeddings.append(pooled.cpu())
        smiles_list.extend(smiles)
        index_list.extend(idxs)

    embeddings = torch.cat(embeddings, dim=0)

    return embeddings, smiles_list, index_list

In [ ]:
import torch

val_data = torch.load("targetdiff/val.pt")

print(len(val_data))
print(val_data[0])

In [ ]:
embeddings, smiles, indices = embed_dataset(val_data, batch_size=8)

print(embeddings.shape)  # (16000, 1024)
print(len(smiles))       # 16000

In [ ]:
torch.save({
    "embeddings": embeddings,
    "smiles": smiles,
    "index": indices
}, "val_embeddings.pt")

In [ ]:
import torch

train_data = torch.load("targetdiff/train.pt")

print(len(train_data))
print(train_data[0])

In [ ]:
embeddings, smiles, indices = embed_dataset(train_data, batch_size=8)

print(embeddings.shape)  # (16000, 1024)
print(len(smiles))       # 16000

In [ ]:
torch.save({
    "embeddings": embeddings,
    "smiles": smiles,
    "index": indices
}, "train_embeddings.pt")

In [ ]:
import os
import torch
from tqdm import tqdm
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import re
from transformers import T5Tokenizer, T5EncoderModel

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load model
model_name = "Rostlab/prot_t5_xl_uniref50"
tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)
model = T5EncoderModel.from_pretrained(model_name).to(device)
model.eval()
########################################
# preprocess
########################################
def preprocess_sequence(seq, max_len=1024):
    seq = seq.replace(" ", "")
    seq = re.sub(r"[UZOB]", "X", seq)
    seq = seq[:max_len]
    return " ".join(list(seq))
########################################
# embedding
########################################
def get_embedding_from_sequence(sequence):
    seq = preprocess_sequence(sequence)

    inputs = tokenizer(seq, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    emb = outputs.last_hidden_state  # (1, L, 1024)

    # mean pooling
    mask = attention_mask.unsqueeze(-1).expand(emb.size()).float()
    summed = torch.sum(emb * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    pooled = summed / counts  # (1, 1024)

    return pooled.squeeze(0).cpu()
########################################
# 1. PDB → sequence
########################################
def pdb_to_sequence(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)

    sequence = ""

    for model in structure:
        for chain in model:
            residues = []
            for res in chain:
                if res.id[0] == " ":
                    residues.append(res.resname)

            sequence += "".join([seq1(r) for r in residues])

            break
        break

    return sequence

########################################
# 2. load ProtT5 (فرض: قبلاً ساختی)
########################################
# tokenizer, model, device, preprocess_sequence
# get_embedding_from_sequence هم قبلاً داری

########################################
# 3. main extraction
########################################
def build_test_embeddings(test_dir):

    results = []

    folders = sorted(os.listdir(test_dir))

    for folder in tqdm(folders):
        path = os.path.join(test_dir, folder)

        if not os.path.isdir(path):
            continue

        # پیدا کردن pdb
        pdb_file = None
        for f in os.listdir(path):
            if f.endswith(".pdb"):
                pdb_file = os.path.join(path, f)
                break

        if pdb_file is None:
            continue

        try:
            # 🔥 sequence
            seq = pdb_to_sequence(pdb_file)

            if len(seq) == 0:
                continue

            # 🔥 embedding
            emb = get_embedding_from_sequence(seq)  # (1024,)

            results.append({
                "id": folder,
                "sequence": seq,
                "embedding": emb
            })

        except Exception as e:
            print("Error in", folder, e)

    return results

In [ ]:
test_dir = "test_set"

data = build_test_embeddings(test_dir)

In [ ]:
torch.save(data, "test_embeddings.pt")

print("Saved:", len(data))